In [2]:
import sympy as sp
import sympy.vector as sv
from IPython.display import display
import enum
from typing import Literal

In [3]:
uT, uB, uL, uR, uc, u_ext = sp.symbols("u_T u_B u_L u_R u_c u_ext")
uT_p, uB_p, uL_p, uR_p, uc_p, u_ext_p = sp.symbols(
    "u_T^+ u_B^+ u_L^+ u_R^+ u_c^+ u_ext^+"
)
uT_m, uB_m, uL_m, uR_m, uc_m, u_ext_m = sp.symbols(
    "u_T^- u_B^- u_L^- u_R^- u_c^- u_ext^-"
)
u = []
for i in range(3):
    u.append(
        [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(3)]
        + [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(-2, 0)]
    )
for i in range(-2, 0):
    u.append(
        [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(3)]
        + [sp.symbols("u_{i%+d\\,j%+d}" % (i, j)) for j in range(-2, 0)]
    )


def flatten(u):
    return [item for sublist in u for item in sublist]


x_i, x_im1, x_ip1 = sp.symbols("x_i x_(i-1) x_(i+1)")
y_j, y_jm1, y_jp1 = sp.symbols("y_j y_(j-1) y_(j+1)")
dx, dy = sp.Symbol(r"\Delta x"), sp.Symbol(r"\Delta y")
xL, xR = sp.symbols("x_L x_R")
yT, yB = sp.symbols("y_T y_B")
# p, pL, pR, pT, pB = sp.symbols(
#     "\\mathbf{p} \\mathbf{p}_L \\mathbf{p}_R \\mathbf{p}_T \\mathbf{p}_B"
# )
x_ext, y_ext = sp.symbols("x_ext y_ext")
theta_L, theta_R, theta_T, theta_B = sp.symbols("theta_L theta_R theta_T theta_B")
coord = sv.CoordSys3D("coord")
x, y = coord.x, coord.y

nx, ny = sp.symbols("n_x n_y")
a, a_tau, b = sp.symbols("a, a_{\\tau}, b")
beta_jump, beta_p, beta_m = sp.symbols("[\\beta], beta^+, beta^-")
# nx, ny = sp.symbols("n_x n_y", cls=sp.Function)
# a, a_tau, b = sp.symbols("a, a_{\\tau}, b", cls=sp.Function)

A = sp.Matrix(
    [
        [x_i**2, x_i * yT, yT**2, x_i, yT, 1],
        [x_i**2, x_i * yB, yB**2, x_i, yB, 1],
        [xL**2, xL * y_j, y_j**2, xL, y_j, 1],
        [xR**2, xR * y_j, y_j**2, xR, y_j, 1],
        [x_i**2, x_i * y_j, y_j**2, x_i, y_j, 1],
        [x_ext**2, x_ext * y_ext, y_ext**2, x_ext, y_ext, 1],
    ]
)
A_inv = A.inv()
P_coeff = A_inv @ sp.Matrix([uT, uB, uL, uR, uc, u_ext])
A, B, C, D, E, F = P_coeff
P = A * x**2 + B * x * y + C * y**2 + D * x + E * y + F

In [4]:
class Direction(enum.IntFlag):
    R = 1 << 0
    T = 1 << 1
    L = 1 << 2
    B = 1 << 3


def coeff_case1(
    eta: Literal[1] | Literal[-1], dir: Direction, vars_m: dict, vars_p: dict
):
    grad_P = sv.gradient(P)
    dudx = grad_P.components[coord.i]
    dudy = grad_P.components[coord.j]

    if dir == Direction.L or dir == Direction.R:
        # geometric discretization of [\beta u_x]
        if eta < 0:
            beta_ux_jump_geometry = (
                b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
            ).subs(vars_m)
        else:
            beta_ux_jump_geometry = (
                b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_m * a_tau * ny
            ).subs(vars_p)
        # algebraic definition of [\beta u_x]
        beta_ux_jump_algebra = beta_p * dudx.subs(vars_p) - beta_m * dudx.subs(vars_m)
        # equate the two definitions
        equality = beta_ux_jump_algebra - beta_ux_jump_geometry
    else:
        # geometric discretization of [\beta u_y]
        if eta < 0:
            beta_uy_jump_geometry = (
                b * ny + beta_jump * nx * (-ny * dudx + nx * dudy) + beta_p * a_tau * nx
            ).subs(vars_m)
        else:
            beta_uy_jump_geometry = (
                b * ny + beta_jump * nx * (-ny * dudx + nx * dudy) + beta_m * a_tau * nx
            ).subs(vars_p)

        # algebraic definition of [\beta u_y]
        beta_uy_jump_algebra = beta_p * dudy.subs(vars_p) - beta_m * dudy.subs(vars_m)
        # equate the two definitions
        equality = beta_uy_jump_algebra - beta_uy_jump_geometry

    if eta < 0:
        if dir == Direction.R:
            u_m = uR_m
            eq_sub = equality.subs({uR_p: u_m + a}).expand().collect(u_m)
        elif dir == Direction.L:
            u_m = uL_m
            eq_sub = equality.subs({uL_p: u_m + a}).expand().collect(u_m)
        elif dir == Direction.T:
            u_m = uT_m
            eq_sub = equality.subs({uT_p: u_m + a}).expand().collect(u_m)
        elif dir == Direction.B:
            u_m = uB_m
            eq_sub = equality.subs({uB_p: u_m + a}).expand().collect(u_m)

        u_m_coeff = eq_sub.coeff(u_m).simplify()
        u_m_coeff = u_m_coeff.collect([beta_p, beta_m, beta_jump])
        tops = u_m_coeff.as_numer_denom()[0].as_ordered_terms()
        bot = u_m_coeff.as_numer_denom()[1]
        rest = (eq_sub - u_m_coeff * u_m).simplify().expand()
    else:
        if dir == Direction.R:
            u_p = uR_p
            eq_sub = equality.subs({uR_m: u_p - a}).expand().collect(u_p)
        elif dir == Direction.L:
            u_p = uL_p
            eq_sub = equality.subs({uL_m: u_p - a}).expand().collect(u_p)
        elif dir == Direction.T:
            u_p = uT_p
            eq_sub = equality.subs({uT_m: u_p - a}).expand().collect(u_p)
        elif dir == Direction.B:
            u_p = uB_p
            eq_sub = equality.subs({uB_m: u_p - a}).expand().collect(u_p)

        u_p_coeff = eq_sub.coeff(u_p).simplify()
        u_p_coeff = u_p_coeff.collect([beta_p, beta_m, beta_jump])
        tops = u_p_coeff.as_numer_denom()[0].as_ordered_terms()
        bot = u_p_coeff.as_numer_denom()[1]
        rest = (eq_sub - u_p_coeff * u_p).simplify().expand()
    # M
    M = sp.Add(*[(top / bot).cancel().factor() for top in tops])

    # separate terms in 'rest' into those involving u variables and those not
    u_terms = []
    non_u_terms = []
    for term in rest.as_ordered_terms():
        if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
            u_terms.append(term)
        else:
            non_u_terms.append(term)

    d = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
        ]
    )
    Nu = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
        ]
    )

    if dir == Direction.L or dir == Direction.R:
        h = dx
    else:
        h = dy
    print("M")
    display(sp.Add(*[(term * h).cancel().factor() for term in M.as_ordered_terms()]))
    print("d")
    display(sp.Add(*[(term * h).cancel().factor() for term in d.as_ordered_terms()]))
    print("Nu")
    display(
        sp.Add(
            *[
                (term * h).cancel().factor()
                for term in Nu.expand().collect(flatten(u)).as_ordered_terms()
            ]
        ).collect(flatten(u))
    )

### case 1, $\eta < 0$ 

In [4]:
case1_right_vars_m = {
    x: x_i + theta_R * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: u[0][-1],
    uT: u[0][1],
    u_ext: u[-1][-1],
}

case1_right_vars_p = {
    x: x_i - (1 - theta_R) * dx,
    y: y_j,
    xL: x_i - (1 - theta_R) * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[1][0],
    uL: uR_p,
    uR: u[2][0],
    uB: u[1][-1],
    uT: u[1][1],
    u_ext: u[1][-1],
}
coeff_case1(-1, Direction.R, vars_m=case1_right_vars_m, vars_p=case1_right_vars_p)

M


-[\beta]*n_y**2*(2*theta_R + 1)/(theta_R*(theta_R + 1)) + beta^+*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1)) - beta^-*(2*theta_R + 1)/(theta_R*(theta_R + 1))

d


-\Delta x*a_{\tau}*beta^+*n_y + \Delta x*b*n_x - a*beta^+*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1))

Nu


-[\beta]*\Delta x*n_x*n_y*theta_R*u_{i-1,j-1}/\Delta y - [\beta]*\Delta x*n_x*n_y*u_{i+0,j+1}/(2*\Delta y) + [\beta]*\Delta x*n_x*n_y*u_{i+0,j-1}*(2*theta_R + 1)/(2*\Delta y) - beta^+*u_{i+1,j+0}*(theta_R - 2)/(theta_R - 1) + beta^+*u_{i+2,j+0}*(theta_R - 1)/(theta_R - 2) + theta_R*u_{i-1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-)/(\Delta y*(theta_R + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R**2 + [\beta]*\Delta y*n_y**2*theta_R + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-*theta_R + \Delta y*beta^-)/(\Delta y*theta_R)

In [5]:
case1_top_vars_m = {
    x: x_i,
    y: y_j + theta_T * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: u[1][0],
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[-1][-1],
}

case1_top_vars_p = {
    x: x_i,
    y: y_j - (1 - theta_T) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - (1 - theta_T) * dy,
    x_ext: x_i - dx,
    y_ext: y_j + dy,
    uc: u[0][1],
    uL: u[-1][1],
    uR: u[1][1],
    uB: uT_p,
    uT: u[0][2],
    u_ext: u[-1][2],
}
coeff_case1(-1, Direction.T, vars_m=case1_top_vars_m, vars_p=case1_top_vars_p)

M


-[\beta]*n_x**2*(2*theta_T + 1)/(theta_T*(theta_T + 1)) + beta^+*(2*theta_T - 3)/((theta_T - 2)*(theta_T - 1)) - beta^-*(2*theta_T + 1)/(theta_T*(theta_T + 1))

d


\Delta y*a_{\tau}*beta^+*n_x + \Delta y*b*n_y - a*beta^+*(2*theta_T - 3)/((theta_T - 2)*(theta_T - 1))

Nu


-[\beta]*\Delta y*n_x*n_y*theta_T*u_{i-1,j-1}/\Delta x - [\beta]*\Delta y*n_x*n_y*u_{i+1,j+0}/(2*\Delta x) + [\beta]*\Delta y*n_x*n_y*u_{i-1,j+0}*(2*theta_T + 1)/(2*\Delta x) - beta^+*u_{i+0,j+1}*(theta_T - 2)/(theta_T - 1) + beta^+*u_{i+0,j+2}*(theta_T - 1)/(theta_T - 2) + theta_T*u_{i+0,j-1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_T + [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*(theta_T + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_T + [\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_T**2 + \Delta x*beta^-*theta_T + \Delta x*beta^-)/(\Delta x*theta_T)

In [6]:
case1_left_vars_m = {
    x: x_i - theta_L * dx,
    y: y_j,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: u[0][-1],
    uT: u[0][1],
    u_ext: u[1][1],
}

case1_left_vars_p = {
    x: x_i + (1 - theta_L) * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + (1 - theta_L) * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j + dy,
    uc: u[-1][0],
    uL: u[-2][0],
    uR: uL_p,
    uB: u[-1][-1],
    uT: u[-1][1],
    u_ext: u[-2][1],
}

coeff_case1(-1, Direction.L, vars_m=case1_left_vars_m, vars_p=case1_left_vars_p)

M


[\beta]*n_y**2*(2*theta_L + 1)/(theta_L*(theta_L + 1)) - beta^+*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1)) + beta^-*(2*theta_L + 1)/(theta_L*(theta_L + 1))

d


-\Delta x*a_{\tau}*beta^+*n_y + \Delta x*b*n_x + a*beta^+*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1))

Nu


[\beta]*\Delta x*n_x*n_y*theta_L*u_{i+1,j+1}/\Delta y - [\beta]*\Delta x*n_x*n_y*u_{i+0,j+1}*(2*theta_L + 1)/(2*\Delta y) + [\beta]*\Delta x*n_x*n_y*u_{i+0,j-1}/(2*\Delta y) + beta^+*u_{i-1,j+0}*(theta_L - 2)/(theta_L - 1) - beta^+*u_{i-2,j+0}*(theta_L - 1)/(theta_L - 2) - theta_L*u_{i+1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-)/(\Delta y*(theta_L + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L**2 + [\beta]*\Delta y*n_y**2*theta_L + [\beta]*\Delta y*n_y**2 + \Delta y*beta^-*theta_L + \Delta y*beta^-)/(\Delta y*theta_L)

In [7]:
case1_bottom_vars_m = {
    x: x_i,
    y: y_j - theta_B * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: u[1][0],
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[1][1],
}

case1_bottom_vars_p = {
    x: x_i,
    y: y_j + (1 - theta_B) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + (1 - theta_B) * dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[0][-1],
    uL: u[-1][-1],
    uR: u[1][-1],
    uB: u[0][-2],
    uT: uB_p,
    u_ext: u[1][-1],
}

coeff_case1(-1, Direction.B, vars_m=case1_bottom_vars_m, vars_p=case1_bottom_vars_p)

M


[\beta]*n_x**2*(2*theta_B + 1)/(theta_B*(theta_B + 1)) - beta^+*(2*theta_B - 3)/((theta_B - 2)*(theta_B - 1)) + beta^-*(2*theta_B + 1)/(theta_B*(theta_B + 1))

d


\Delta y*a_{\tau}*beta^+*n_x + \Delta y*b*n_y + a*beta^+*(2*theta_B - 3)/((theta_B - 2)*(theta_B - 1))

Nu


[\beta]*\Delta y*n_x*n_y*theta_B*u_{i+1,j+1}/\Delta x - [\beta]*\Delta y*n_x*n_y*u_{i+1,j+0}*(2*theta_B + 1)/(2*\Delta x) + [\beta]*\Delta y*n_x*n_y*u_{i-1,j+0}/(2*\Delta x) + beta^+*u_{i+0,j-1}*(theta_B - 2)/(theta_B - 1) - beta^+*u_{i+0,j-2}*(theta_B - 1)/(theta_B - 2) - theta_B*u_{i+0,j+1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_B + [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*(theta_B + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_B + [\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_B**2 + \Delta x*beta^-*theta_B + \Delta x*beta^-)/(\Delta x*theta_B)

### Case 1, $\eta > 0$

In [8]:
case1_right_vars_p = {
    x: x_i + theta_R * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_p,
    uB: u[0][-1],
    uT: u[0][1],
    u_ext: u[-1][-1],
}

case1_right_vars_m = {
    x: x_i - (1 - theta_R) * dx,
    y: y_j,
    xL: x_i - (1 - theta_R) * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[1][0],
    uL: uR_m,
    uR: u[2][0],
    uB: u[1][-1],
    uT: u[1][1],
    u_ext: u[1][-1],
}
coeff_case1(1, Direction.R, vars_m=case1_right_vars_m, vars_p=case1_right_vars_p)

M


-[\beta]*n_y**2*(2*theta_R + 1)/(theta_R*(theta_R + 1)) + beta^+*(2*theta_R + 1)/(theta_R*(theta_R + 1)) - beta^-*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1))

d


-\Delta x*a_{\tau}*beta^-*n_y + \Delta x*b*n_x - a*beta^-*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1))

Nu


-[\beta]*\Delta x*n_x*n_y*theta_R*u_{i-1,j-1}/\Delta y - [\beta]*\Delta x*n_x*n_y*u_{i+0,j+1}/(2*\Delta y) + [\beta]*\Delta x*n_x*n_y*u_{i+0,j-1}*(2*theta_R + 1)/(2*\Delta y) + beta^-*u_{i+1,j+0}*(theta_R - 2)/(theta_R - 1) - beta^-*u_{i+2,j+0}*(theta_R - 1)/(theta_R - 2) + theta_R*u_{i-1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 - \Delta y*beta^+)/(\Delta y*(theta_R + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_R**2 + [\beta]*\Delta y*n_y**2*theta_R + [\beta]*\Delta y*n_y**2 - \Delta y*beta^+*theta_R - \Delta y*beta^+)/(\Delta y*theta_R)

In [9]:
case1_top_vars_p = {
    x: x_i,
    y: y_j + theta_T * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: u[1][0],
    uB: u[0][-1],
    uT: uT_p,
    u_ext: u[-1][-1],
}

case1_top_vars_m = {
    x: x_i,
    y: y_j - (1 - theta_T) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - (1 - theta_T) * dy,
    x_ext: x_i - dx,
    y_ext: y_j + dy,
    uc: u[0][1],
    uL: u[-1][1],
    uR: u[1][1],
    uB: uT_m,
    uT: u[0][2],
    u_ext: u[-1][2],
}
coeff_case1(1, Direction.T, vars_m=case1_top_vars_m, vars_p=case1_top_vars_p)

M


-[\beta]*n_x**2*(2*theta_T + 1)/(theta_T*(theta_T + 1)) + beta^+*(2*theta_T + 1)/(theta_T*(theta_T + 1)) - beta^-*(2*theta_T - 3)/((theta_T - 2)*(theta_T - 1))

d


\Delta y*a_{\tau}*beta^-*n_x + \Delta y*b*n_y - a*beta^-*(2*theta_T - 3)/((theta_T - 2)*(theta_T - 1))

Nu


-[\beta]*\Delta y*n_x*n_y*theta_T*u_{i-1,j-1}/\Delta x - [\beta]*\Delta y*n_x*n_y*u_{i+1,j+0}/(2*\Delta x) + [\beta]*\Delta y*n_x*n_y*u_{i-1,j+0}*(2*theta_T + 1)/(2*\Delta x) + beta^-*u_{i+0,j+1}*(theta_T - 2)/(theta_T - 1) - beta^-*u_{i+0,j+2}*(theta_T - 1)/(theta_T - 2) + theta_T*u_{i+0,j-1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_T + [\beta]*\Delta y*n_x*n_y - \Delta x*beta^+)/(\Delta x*(theta_T + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_T + [\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_T**2 - \Delta x*beta^+*theta_T - \Delta x*beta^+)/(\Delta x*theta_T)

In [10]:
case1_left_vars_p = {
    x: x_i - theta_L * dx,
    y: y_j,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: uL_p,
    uR: u[1][0],
    uB: u[0][-1],
    uT: u[0][1],
    u_ext: u[1][1],
}

case1_left_vars_m = {
    x: x_i + (1 - theta_L) * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + (1 - theta_L) * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j + dy,
    uc: u[-1][0],
    uL: u[-2][0],
    uR: uL_m,
    uB: u[-1][-1],
    uT: u[-1][1],
    u_ext: u[-2][1],
}

coeff_case1(1, Direction.L, vars_m=case1_left_vars_m, vars_p=case1_left_vars_p)

M


[\beta]*n_y**2*(2*theta_L + 1)/(theta_L*(theta_L + 1)) - beta^+*(2*theta_L + 1)/(theta_L*(theta_L + 1)) + beta^-*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1))

d


-\Delta x*a_{\tau}*beta^-*n_y + \Delta x*b*n_x + a*beta^-*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1))

Nu


[\beta]*\Delta x*n_x*n_y*theta_L*u_{i+1,j+1}/\Delta y - [\beta]*\Delta x*n_x*n_y*u_{i+0,j+1}*(2*theta_L + 1)/(2*\Delta y) + [\beta]*\Delta x*n_x*n_y*u_{i+0,j-1}/(2*\Delta y) - beta^-*u_{i-1,j+0}*(theta_L - 2)/(theta_L - 1) + beta^-*u_{i-2,j+0}*(theta_L - 1)/(theta_L - 2) - theta_L*u_{i+1,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L + [\beta]*\Delta x*n_x*n_y + [\beta]*\Delta y*n_y**2 - \Delta y*beta^+)/(\Delta y*(theta_L + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x*n_y*theta_L**2 + [\beta]*\Delta y*n_y**2*theta_L + [\beta]*\Delta y*n_y**2 - \Delta y*beta^+*theta_L - \Delta y*beta^+)/(\Delta y*theta_L)

In [11]:
case1_bottom_vars_p = {
    x: x_i,
    y: y_j - theta_B * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: u[1][0],
    uB: uB_p,
    uT: u[0][1],
    u_ext: u[1][1],
}

case1_bottom_vars_m = {
    x: x_i,
    y: y_j + (1 - theta_B) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + (1 - theta_B) * dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[0][-1],
    uL: u[-1][-1],
    uR: u[1][-1],
    uB: u[0][-2],
    uT: uB_m,
    u_ext: u[1][-1],
}

coeff_case1(1, Direction.B, vars_m=case1_bottom_vars_m, vars_p=case1_bottom_vars_p)

M


[\beta]*n_x**2*(2*theta_B + 1)/(theta_B*(theta_B + 1)) - beta^+*(2*theta_B + 1)/(theta_B*(theta_B + 1)) + beta^-*(2*theta_B - 3)/((theta_B - 2)*(theta_B - 1))

d


\Delta y*a_{\tau}*beta^-*n_x + \Delta y*b*n_y + a*beta^-*(2*theta_B - 3)/((theta_B - 2)*(theta_B - 1))

Nu


[\beta]*\Delta y*n_x*n_y*theta_B*u_{i+1,j+1}/\Delta x - [\beta]*\Delta y*n_x*n_y*u_{i+1,j+0}*(2*theta_B + 1)/(2*\Delta x) + [\beta]*\Delta y*n_x*n_y*u_{i-1,j+0}/(2*\Delta x) - beta^-*u_{i+0,j-1}*(theta_B - 2)/(theta_B - 1) + beta^-*u_{i+0,j-2}*(theta_B - 1)/(theta_B - 2) - theta_B*u_{i+0,j+1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_B + [\beta]*\Delta y*n_x*n_y - \Delta x*beta^+)/(\Delta x*(theta_B + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_B + [\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_B**2 - \Delta x*beta^+*theta_B - \Delta x*beta^+)/(\Delta x*theta_B)

### Case 2

In [5]:
def coeff_case2(
    eta: Literal[1, -1],
    direction: int,
    vars_x_p: dict,
    vars_x_m: dict,
    vars_y_p: dict,
    vars_y_m: dict,
):

    grad_P = sv.gradient(P)
    dudx = grad_P.components[coord.i]
    dudy = grad_P.components[coord.j]

    if eta < 0:
        beta_ux_jump_geometry = (
            b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_p * a_tau * ny
        ).subs(vars_x_m)
    else:
        beta_ux_jump_geometry = (
            b * nx - beta_jump * ny * (-ny * dudx + nx * dudy) - beta_m * a_tau * ny
        ).subs(vars_x_p)
    beta_ux_jump_algebra = beta_p * dudx.subs(vars_x_p) - beta_m * dudx.subs(vars_x_m)
    equality_x = beta_ux_jump_algebra - beta_ux_jump_geometry
    equality_x *= dx

    if eta < 0:
        beta_uy_jump_geometry = (
            b * ny + beta_jump * nx * (-ny * dudx + nx * dudy) + beta_p * a_tau * nx
        ).subs(vars_y_m)
    else:
        beta_uy_jump_geometry = (
            b * ny + beta_jump * nx * (-ny * dudx + nx * dudy) + beta_m * a_tau * nx
        ).subs(vars_y_p)
    beta_uy_jump_algebra = beta_p * dudy.subs(vars_y_p) - beta_m * dudy.subs(vars_y_m)
    equality_y = beta_uy_jump_algebra - beta_uy_jump_geometry
    equality_y *= dy

    # M
    M = sp.Matrix([[0, 0], [0, 0]])
    Nu = sp.Matrix([0, 0])
    d = sp.Matrix([0, 0])
    # match statement doesn't work since | in case is not bit-wise or operator
    if direction == Direction.R | Direction.T:
        ux_m = uR_m
        uy_m = uT_m
        eq_sub_x = equality_x.subs({uR_p: ux_m + a, uT_p: uy_m + a}).expand()
        eq_sub_y = equality_y.subs({uR_p: ux_m + a, uT_p: uy_m + a}).expand()
    elif direction == Direction.L | Direction.T:
        ux_m = uL_m
        uy_m = uT_m
        eq_sub_x = equality_x.subs({uL_p: ux_m + a, uT_p: uy_m + a}).expand()
        eq_sub_y = equality_y.subs({uL_p: ux_m + a, uT_p: uy_m + a}).expand()
    elif direction == Direction.R | Direction.B:
        ux_m = uR_m
        uy_m = uB_m
        eq_sub_x = equality_x.subs({uR_p: ux_m + a, uB_p: uy_m + a}).expand()
        eq_sub_y = equality_y.subs({uR_p: ux_m + a, uB_p: uy_m + a}).expand()
    elif direction == Direction.L | Direction.B:
        ux_m = uL_m
        uy_m = uB_m
        eq_sub_x = equality_x.subs({uL_p: ux_m + a, uB_p: uy_m + a}).expand()
        eq_sub_y = equality_y.subs({uL_p: ux_m + a, uB_p: uy_m + a}).expand()
    else:
        raise ValueError("No such direction...", direction)

    ux_m_coeff_x = eq_sub_x.coeff(ux_m).simplify().collect([beta_p, beta_m, beta_jump])
    uy_m_coeff_x = eq_sub_x.coeff(uy_m).simplify().collect([beta_p, beta_m, beta_jump])
    rest_x = (eq_sub_x - ux_m_coeff_x * ux_m - uy_m_coeff_x * uy_m).simplify().expand()

    ux_m_coeff_y = eq_sub_y.coeff(ux_m).simplify().collect([beta_p, beta_m, beta_jump])
    uy_m_coeff_y = eq_sub_y.coeff(uy_m).simplify().collect([beta_p, beta_m, beta_jump])
    rest_y = (eq_sub_y - ux_m_coeff_y * ux_m - uy_m_coeff_y * uy_m).simplify().expand()

    M[0, 0] = sp.Add(
        *[
            (top / ux_m_coeff_x.as_numer_denom()[1]).cancel().factor()
            for top in ux_m_coeff_x.as_numer_denom()[0].as_ordered_terms()
        ]
    )

    M[0, 1] = sp.Add(
        *[
            (top / uy_m_coeff_x.as_numer_denom()[1]).cancel().factor()
            for top in uy_m_coeff_x.as_numer_denom()[0].as_ordered_terms()
        ]
    )

    M[1, 0] = sp.Add(
        *[
            (top / ux_m_coeff_y.as_numer_denom()[1]).cancel().factor()
            for top in ux_m_coeff_y.as_numer_denom()[0].as_ordered_terms()
        ]
    )

    M[1, 1] = sp.Add(
        *[
            (top / uy_m_coeff_y.as_numer_denom()[1]).cancel().factor()
            for top in uy_m_coeff_y.as_numer_denom()[0].as_ordered_terms()
        ]
    )

    u_terms = []
    non_u_terms = []
    for term in rest_x.as_ordered_terms():
        if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
            u_terms.append(term)
        else:
            non_u_terms.append(term)

    d[0] = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
        ]
    )
    Nu[0] = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
        ]
    )

    u_terms = []
    non_u_terms = []
    for term in rest_y.as_ordered_terms():
        if any(u_var in term.free_symbols for sublist in u for u_var in sublist):
            u_terms.append(term)
        else:
            non_u_terms.append(term)

    d[1] = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*non_u_terms).collect([a, b, a_tau]).as_ordered_terms()
        ]
    )
    Nu[1] = -sp.Add(
        *[
            term.cancel().factor()
            for term in sp.Add(*u_terms).collect(flatten(u)).as_ordered_terms()
        ]
    )
    print("M[0,0]")
    display(M[0, 0])
    print("M[0,1]")
    display(M[0, 1])
    print("M[1,0]")
    display(M[1, 0])
    print("M[1,1]")
    display(M[1, 1])

    print("Nu[0]")
    display(Nu[0])
    print("Nu[1]")
    display(Nu[1])

    print("d[0]")
    display(d[0])
    print("d[1]")
    display(d[1])

In [13]:
case2_right_vars_m = { # top right
    x: x_i + theta_R * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[-1][-1],
}

case2_top_vars_m = {
    x: x_i,
    y: y_j + theta_T * dy,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[-1][-1],
}

case2_right_vars_p = {
    x: x_i - (1 - theta_R) * dx,
    y: y_j,
    xL: x_i - (1 - theta_R) * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[1][0],
    uL: uR_p,
    uR: u[2][0],
    uB: u[1][-1],
    uT: u[1][1],
    u_ext: u[1][-1],
}

case2_top_vars_p = {
    x: x_i,
    y: y_j - (1 - theta_T) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - (1 - theta_T) * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][1],
    uL: u[-1][1],
    uR: u[1][1],
    uB: uT_p,
    uT: u[0][2],
    u_ext: u[2][2],
}

coeff_case2(
    -1,
    Direction.R | Direction.T,
    vars_x_p=case2_right_vars_p,
    vars_x_m=case2_right_vars_m,
    vars_y_p=case2_top_vars_p,
    vars_y_m=case2_top_vars_m,
)

M[0,0]


-[\beta]*n_y**2*(2*theta_R + 1)/(theta_R*(theta_R + 1)) + beta^+*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1)) - beta^-*(2*theta_R + 1)/(theta_R*(theta_R + 1))

M[0,1]


[\beta]*\Delta x*n_x*n_y/(\Delta y*theta_T*(theta_T + 1))

M[1,0]


...

-[\beta]*\Delta y*n_x*n_y*theta_T*u_{i-1,j-1}/\Delta x + [\beta]*\Delta y*n_x*n_y*u_{i-1,j+0}*(theta_R*theta_T + theta_R + theta_T)/(\Delta x*(theta_R + 1)) - beta^+*u_{i+0,j+1}*(theta_T - 2)/(theta_T - 1) + beta^+*u_{i+0,j+2}*(theta_T - 1)/(theta_T - 2) + theta_T*u_{i+0,j-1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_T + [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*(theta_T + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_R*theta_T + [\beta]*\Delta x*n_x**2*theta_R + [\beta]*\Delta y*n_x*n_y*theta_R*theta_T**2 + [\beta]*\Delta y*n_x*n_y*theta_R*theta_T - [\beta]*\Delta y*n_x*n_y*theta_T + \Delta x*beta^-*theta_R*theta_T + \Delta x*beta^-*theta_R)/(\Delta x*theta_R*theta_T)

d[0]


-\Delta x*a_{\tau}*beta^+*n_y + \Delta x*b*n_x - a*beta^+*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1))

d[1]


\Delta y*a_{\tau}*beta^+*n_x + \Delta y*b*n_y - a*beta^+*(2*theta_T - 3)/((theta_T - 2)*(theta_T - 1))

In [14]:
case2_left_vars_m = {  # left top
    x: x_i - theta_L * dx,
    y: y_j,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[1][-1],
}

case2_top_vars_m = {
    x: x_i,
    y: y_j + theta_T * dy,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + theta_T * dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: u[0][-1],
    uT: uT_m,
    u_ext: u[1][-1],
}

case2_left_vars_p = {
    x: x_i + (1 - theta_L) * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + (1 - theta_L) * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[-1][0],
    uL: u[-2][0],
    uR: uL_p,
    uB: u[-1][-1],
    uT: u[-1][1],
    u_ext: u[-1][-1],
}

case2_top_vars_p = {
    x: x_i,
    y: y_j - (1 - theta_T) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - (1 - theta_T) * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][1],
    uL: u[-1][1],
    uR: u[1][1],
    uB: uT_p,
    uT: u[0][2],
    u_ext: u[2][2],
}

coeff_case2(
    -1,
    Direction.L | Direction.T,
    vars_x_p=case2_left_vars_p,
    vars_x_m=case2_left_vars_m,
    vars_y_p=case2_top_vars_p,
    vars_y_m=case2_top_vars_m,
)

M[0,0]


[\beta]*n_y**2*(2*theta_L + 1)/(theta_L*(theta_L + 1)) - beta^+*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1)) + beta^-*(2*theta_L + 1)/(theta_L*(theta_L + 1))

M[0,1]


[\beta]*\Delta x*n_x*n_y/(\Delta y*theta_T*(theta_T + 1))

M[1,0]


...

[\beta]*\Delta y*n_x*n_y*theta_T*u_{i+1,j-1}/\Delta x - [\beta]*\Delta y*n_x*n_y*u_{i+1,j+0}*(theta_L*theta_T + theta_L + theta_T)/(\Delta x*(theta_L + 1)) - beta^+*u_{i+0,j+1}*(theta_T - 2)/(theta_T - 1) + beta^+*u_{i+0,j+2}*(theta_T - 1)/(theta_T - 2) + theta_T*u_{i+0,j-1}*([\beta]*\Delta x*n_x**2 - [\beta]*\Delta y*n_x*n_y*theta_T - [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*(theta_T + 1)) - u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_L*theta_T + [\beta]*\Delta x*n_x**2*theta_L - [\beta]*\Delta y*n_x*n_y*theta_L*theta_T**2 - [\beta]*\Delta y*n_x*n_y*theta_L*theta_T + [\beta]*\Delta y*n_x*n_y*theta_T + \Delta x*beta^-*theta_L*theta_T + \Delta x*beta^-*theta_L)/(\Delta x*theta_L*theta_T)

d[0]


-\Delta x*a_{\tau}*beta^+*n_y + \Delta x*b*n_x + a*beta^+*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1))

d[1]


\Delta y*a_{\tau}*beta^+*n_x + \Delta y*b*n_y - a*beta^+*(2*theta_T - 3)/((theta_T - 2)*(theta_T - 1))

In [15]:
case2_left_vars_m = { # left bottom
    x: x_i - theta_L * dx,
    y: y_j,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[1][1],
}

case2_bot_vars_m = {
    x: x_i,
    y: y_j - theta_B * dy,
    xL: x_i - theta_L * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i + dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: uL_m,
    uR: u[1][0],
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[1][1],
}

case2_left_vars_p = {
    x: x_i + (1 - theta_L) * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + (1 - theta_L) * dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[-1][0],
    uL: u[-2][0],
    uR: uL_p,
    uB: u[-1][-1],
    uT: u[-1][1],
    u_ext: u[-1][-1],
}

case2_bot_vars_p = {
    x: x_i,
    y: y_j + (1 - theta_B) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + (1 - theta_B) * dy,
    yB: y_j -  dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][-1],
    uL: u[-1][-1],
    uR: u[1][-1],
    uB: u[1][-2],
    uT: uB_p,
    u_ext: u[-2][-2],
}

coeff_case2(
    -1,
    Direction.L | Direction.B,
    vars_x_p=case2_left_vars_p,
    vars_x_m=case2_left_vars_m,
    vars_y_p=case2_bot_vars_p,
    vars_y_m=case2_bot_vars_m,
)

M[0,0]


[\beta]*n_y**2*(2*theta_L + 1)/(theta_L*(theta_L + 1)) - beta^+*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1)) + beta^-*(2*theta_L + 1)/(theta_L*(theta_L + 1))

M[0,1]


-[\beta]*\Delta x*n_x*n_y/(\Delta y*theta_B*(theta_B + 1))

M[1,0]


...

[\beta]*\Delta y*n_x*n_y*theta_B*u_{i+1,j+1}/\Delta x - [\beta]*\Delta y*n_x*n_y*u_{i+1,j+0}*(theta_B*theta_L + theta_B + theta_L)/(\Delta x*(theta_L + 1)) + beta^+*u_{i+0,j-1}*(theta_B - 2)/(theta_B - 1) - beta^+*u_{i+1,j-2}*(theta_B - 1)/(theta_B - 2) - theta_B*u_{i+0,j+1}*([\beta]*\Delta x*n_x**2 + [\beta]*\Delta y*n_x*n_y*theta_B + [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*(theta_B + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_B*theta_L + [\beta]*\Delta x*n_x**2*theta_L + [\beta]*\Delta y*n_x*n_y*theta_B**2*theta_L + [\beta]*\Delta y*n_x*n_y*theta_B*theta_L - [\beta]*\Delta y*n_x*n_y*theta_B + \Delta x*beta^-*theta_B*theta_L + \Delta x*beta^-*theta_L)/(\Delta x*theta_B*theta_L)

d[0]


-\Delta x*a_{\tau}*beta^+*n_y + \Delta x*b*n_x + a*beta^+*(2*theta_L - 3)/((theta_L - 2)*(theta_L - 1))

d[1]


\Delta y*a_{\tau}*beta^+*n_x + \Delta y*b*n_y + a*beta^+*(2*theta_B - 3)/((theta_B - 2)*(theta_B - 1))

In [6]:
case2_right_vars_m = { # right bottom
    x: x_i + theta_R * dx,
    y: y_j,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i - dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[-1][1],
}

case2_bot_vars_m = {
    x: x_i,
    y: y_j - theta_B * dy,
    xL: x_i - dx,
    xR: x_i + theta_R * dx,
    yT: y_j + dy,
    yB: y_j - theta_B * dy,
    x_ext: x_i - dx,
    y_ext: y_j + dy,
    uc: u[0][0],
    uL: u[-1][0],
    uR: uR_m,
    uB: uB_m,
    uT: u[0][1],
    u_ext: u[-1][1],
}

case2_right_vars_p = {
    x: x_i - (1 - theta_R) * dx,
    y: y_j,
    xL: x_i - (1 - theta_R) * dx,
    xR: x_i + dx,
    yT: y_j + dy,
    yB: y_j - dy,
    x_ext: x_i + dx,
    y_ext: y_j - dy,
    uc: u[1][0],
    uL: uR_p,
    uR: u[2][0],
    uB: u[1][-1],
    uT: u[1][1],
    u_ext: u[2][-1],
}

case2_bot_vars_p = {
    x: x_i,
    y: y_j + (1 - theta_B) * dy,
    xL: x_i - dx,
    xR: x_i + dx,
    yT: y_j + (1 - theta_B) * dy,
    yB: y_j -  dy,
    x_ext: x_i - dx,
    y_ext: y_j - dy,
    uc: u[0][-1],
    uL: u[-1][-1],
    uR: u[1][-1],
    uB: u[1][-2],
    uT: uB_p,
    u_ext: u[-1][-2],
}

coeff_case2(
    -1,
    Direction.R | Direction.B,
    vars_x_p=case2_right_vars_p,
    vars_x_m=case2_right_vars_m,
    vars_y_p=case2_bot_vars_p,
    vars_y_m=case2_bot_vars_m,
)

M[0,0]


-[\beta]*n_y**2*(2*theta_R + 1)/(theta_R*(theta_R + 1)) + beta^+*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1)) - beta^-*(2*theta_R + 1)/(theta_R*(theta_R + 1))

M[0,1]


-[\beta]*\Delta x*n_x*n_y/(\Delta y*theta_B*(theta_B + 1))

M[1,0]


...

-[\beta]*\Delta y*n_x*n_y*theta_B*u_{i-1,j+1}/\Delta x + [\beta]*\Delta y*n_x*n_y*u_{i-1,j+0}*(theta_B*theta_R + theta_B + theta_R)/(\Delta x*(theta_R + 1)) + beta^+*u_{i+0,j-1}*(theta_B - 2)/(theta_B - 1) - beta^+*u_{i+1,j-2}*(theta_B - 1)/(theta_B - 2) - theta_B*u_{i+0,j+1}*([\beta]*\Delta x*n_x**2 - [\beta]*\Delta y*n_x*n_y*theta_B - [\beta]*\Delta y*n_x*n_y + \Delta x*beta^-)/(\Delta x*(theta_B + 1)) + u_{i+0,j+0}*([\beta]*\Delta x*n_x**2*theta_B*theta_R + [\beta]*\Delta x*n_x**2*theta_R - [\beta]*\Delta y*n_x*n_y*theta_B**2*theta_R - [\beta]*\Delta y*n_x*n_y*theta_B*theta_R + [\beta]*\Delta y*n_x*n_y*theta_B + \Delta x*beta^-*theta_B*theta_R + \Delta x*beta^-*theta_R)/(\Delta x*theta_B*theta_R)

d[0]


-\Delta x*a_{\tau}*beta^+*n_y + \Delta x*b*n_x - a*beta^+*(2*theta_R - 3)/((theta_R - 2)*(theta_R - 1))

d[1]


\Delta y*a_{\tau}*beta^+*n_x + \Delta y*b*n_y + a*beta^+*(2*theta_B - 3)/((theta_B - 2)*(theta_B - 1))